In [3]:
from pathlib import Path

SOURCE_AUDIO = (
    Path.home() / "vocult" / "files" / "test_audio.mp3"
)

result = emotion_classifier.classify_file(str(SOURCE_AUDIO))

print("Predicted emotion:", result[3])
print("Prediction scores:", result[1])

CategoricalEncoder.expect_len was never called: assuming category count of 4 to be correct! Sanity check your encoder using `.expect_len`. Ensure that downstream code also uses the correct size. If you are sure this does not apply to you, use `.ignore_len`.


Predicted emotion: ['neu']
Prediction scores: tensor([1.0000])


In [9]:
from collections import Counter

smoothed_timeline = []

for i, item in enumerate(emotion_timeline):
    left = max(0, i - 1)
    right = min(len(emotion_timeline), i + 2)

    nearby_labels = [
        x["emotion"] for x in emotion_timeline[left:right]
    ]

    majority_emotion = Counter(nearby_labels).most_common(1)[0][0]

    smoothed_timeline.append({
        **item,
        "emotion": majority_emotion
    })

smoothed_file = OUTPUT_DIR / "emotion_timeline_smoothed.json"

with open(smoothed_file, "w", encoding="utf-8") as f:
    json.dump(smoothed_timeline, f, indent=2, ensure_ascii=False)

print("Smoothed emotion timeline saved to:", smoothed_file)

Smoothed emotion timeline saved to: /Users/chaitanyabhardwaj/vocult/files/emotion_output/emotion_timeline_smoothed.json


In [1]:
import torch
from transformers.pytorch_utils import isin_mps_friendly
from TTS.api import TTS

print("PyTorch:", torch.__version__)
print("XTTS import successful!")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.13.0
XTTS import successful!


In [2]:
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

tts = TTS(
    model_name="tts_models/multilingual/multi-dataset/xtts_v2"
).to(DEVICE)

print("XTTS loaded on:", DEVICE)

XTTS loaded on: mps


In [4]:
import json
from pathlib import Path

BASE_DIR = Path("/Users/chaitanyabhardwaj/vocult/files")

TRANSLATED_JSON = BASE_DIR / "test_audio_text_translated.json"

with open(TRANSLATED_JSON, "r", encoding="utf-8") as f:
    translated_data = json.load(f)

print("Translated JSON loaded successfully!")
print("Data type:", type(translated_data))

if isinstance(translated_data, dict):
    print("Available keys:", translated_data.keys())
elif isinstance(translated_data, list):
    print("Number of segments:", len(translated_data))

print("First item:", translated_data[0] if isinstance(translated_data, list)
      else list(translated_data.items())[:1])

Translated JSON loaded successfully!
Data type: <class 'dict'>
Available keys: dict_keys(['source_language', 'target_language', 'segments', 'metadata'])
First item: [('source_language', 'eng_Latn')]


In [5]:
if isinstance(translated_data, list):
    segments = translated_data
elif isinstance(translated_data, dict):
    segments = translated_data.get("segments", [])

if not segments:
    raise ValueError("No segments found. Check the JSON structure.")

SEGMENTS_DIR = BASE_DIR / "tts_segments"
SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Found {len(segments)} translated segments.")
print("First translated sentence:", segments[0].get("translated_text"))

Found 42 translated segments.
First translated sentence: मनुष्य के रूप में, हम एक दूसरे को समझते हैं।


In [8]:
from pathlib import Path
import subprocess
import shutil

BASE_DIR = Path("/Users/chaitanyabhardwaj/vocult/files")
reference_audio = BASE_DIR / "speaker_reference.wav"
converted_audio = BASE_DIR / "speaker_reference_24k.wav"

if not shutil.which("ffmpeg"):
    raise RuntimeError("FFmpeg is not installed or is not on PATH.")

subprocess.run([
    "ffmpeg", "-y",
    "-i", str(reference_audio),
    "-ac", "1",
    "-ar", "24000",
    str(converted_audio)
], check=True)

print("Converted reference audio:", converted_audio)

Converted reference audio: /Users/chaitanyabhardwaj/vocult/files/speaker_reference_24k.wav


ffmpeg version 9.0.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.3.34.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/9.0.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      61.  1.102 / 61.  1.102
  libavcodec     63.  1.102 / 63.  1.102
  libavformat    63.  1.102 / 63.  1.102
  libavdevice    63.  1.102 / 63.  1.102
  libavfilter    12.  1.102 / 12.  1.102
  libswscale     10.  1.102 / 10.  1.102
  libswresample   7.  1.102 /  7.  1.102
[aist#0:0/pcm_s16le @ 0x7aa7008180] Guessed Channel Layout: mono
Input #0, wav, from '/Users/chaitanyabhardwaj/vocult/files/speaker_reference.wav':
  Metadata:
    artist          : Dee

In [10]:
audio, sample_rate = sf.read(converted_audio, dtype="float32")

print("Audio shape:", audio.shape)
print("Sample rate:", sample_rate)
print("Audio loaded successfully!")%pip install edge-tts soundfile numpy imageio-ffmpeg

Audio shape: (240000,)
Sample rate: 24000
Audio loaded successfully!


In [11]:
%pip install edge-tts soundfile numpy imageio-ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [edge-tts]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import asyncio
import json
import re
import subprocess
import shutil
from pathlib import Path

import edge_tts
import imageio_ffmpeg
import numpy as np
import soundfile as sf

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
BASE_DIR = Path("/Users/chaitanyabhardwaj/vocult/files")

VIDEO_PATH = BASE_DIR / "test_video.mp4"
JSON_PATH = BASE_DIR / "test_audio_text_translated.json"

OUTPUT_DIR = BASE_DIR / "output"
SEGMENTS_DIR = BASE_DIR / "hindi_tts_segments"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_AUDIO = OUTPUT_DIR / "translated_speech_hindi.wav"
FINAL_VIDEO = OUTPUT_DIR / "test_video_hindi.mp4"

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
SAMPLE_RATE = 24000
VOICE = "hi-IN-SwaraNeural"

# --------------------------------------------------
# 2. Validate inputs and load translated JSON
# --------------------------------------------------
if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")

if not JSON_PATH.exists():
    raise FileNotFoundError(f"Translated JSON not found: {JSON_PATH}")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    translated_data = json.load(f)

if isinstance(translated_data, list):
    segments = translated_data
elif isinstance(translated_data, dict):
    segments = translated_data.get("segments", [])
else:
    segments = []

if not segments:
    raise ValueError("No translated segments found in the JSON.")

print("Video:", VIDEO_PATH)
print("Translated segments:", len(segments))
print("Hindi voice:", VOICE)

# --------------------------------------------------
# 3. Get the original video duration
# --------------------------------------------------
probe = subprocess.run(
    [FFMPEG, "-i", str(VIDEO_PATH)],
    capture_output=True,
    text=True
)

match = re.search(
    r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)",
    probe.stderr
)

if not match:
    raise RuntimeError(
        "Could not determine video duration. Check that the MP4 is valid."
    )

video_duration = (
    int(match.group(1)) * 3600
    + int(match.group(2)) * 60
    + float(match.group(3))
)

print(f"Video duration: {video_duration:.2f} seconds")

# --------------------------------------------------
# 4. Generate Hindi speech for every segment
# --------------------------------------------------
async def generate_hindi_segments():
    generated = []

    for i, segment in enumerate(segments):
        text = (
            segment.get("translated_text")
            or segment.get("text")
            or ""
        ).strip()

        if not text:
            continue

        start = float(segment.get("start", 0))
        if start < 0:
            start = 0

        mp3_path = SEGMENTS_DIR / f"segment_{i:03d}.mp3"
        wav_path = SEGMENTS_DIR / f"segment_{i:03d}.wav"

        # Generate Hindi speech
        communicate = edge_tts.Communicate(
            text=text,
            voice=VOICE
        )
        await communicate.save(str(mp3_path))

        # Convert speech to 24 kHz mono WAV
        subprocess.run(
            [
                FFMPEG, "-y",
                "-i", str(mp3_path),
                "-ac", "1",
                "-ar", str(SAMPLE_RATE),
                "-c:a", "pcm_s16le",
                str(wav_path)
            ],
            check=True,
            capture_output=True
        )

        audio, sr = sf.read(wav_path, dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        generated.append({
            "index": i,
            "text": text,
            "start": start,
            "audio": audio
        })

        print(f"Generated {len(generated)}: {text[:65]}")

    return generated


generated_segments = await generate_hindi_segments()

if not generated_segments:
    raise RuntimeError("No speech segments were generated.")

print(f"\nSuccessfully generated {len(generated_segments)} Hindi segments.")

# --------------------------------------------------
# 5. Align generated speech with original timestamps
# --------------------------------------------------
total_samples = int(video_duration * SAMPLE_RATE)
timeline = np.zeros(total_samples, dtype=np.float32)

for item in generated_segments:
    start_sample = int(item["start"] * SAMPLE_RATE)
    audio = item["audio"]

    if start_sample >= total_samples:
        continue

    # Keep audio within the video's duration
    end_sample = min(
        start_sample + len(audio),
        total_samples
    )
    audio = audio[:end_sample - start_sample]

    # Mix segments at their original start times
    timeline[start_sample:end_sample] += audio

# Prevent clipping after mixing
peak = np.max(np.abs(timeline))
if peak > 0.99:
    timeline = timeline * (0.99 / peak)

sf.write(FINAL_AUDIO, timeline, SAMPLE_RATE)

print("Hindi audio saved:", FINAL_AUDIO)

# --------------------------------------------------
# 6. Merge Hindi audio with the original video
# --------------------------------------------------
subprocess.run(
    [
        FFMPEG, "-y",
        "-i", str(VIDEO_PATH),
        "-i", str(FINAL_AUDIO),
        "-map", "0:v:0",
        "-map", "1:a:0",
        "-c:v", "copy",
        "-c:a", "aac",
        "-b:a", "192k",
        "-t", str(video_duration),
        "-movflags", "+faststart",
        str(FINAL_VIDEO)
    ],
    check=True
)

print("\nHindi dubbing complete!")
print("Final video:", FINAL_VIDEO)
print("Video exists:", FINAL_VIDEO.exists())


Video: /Users/chaitanyabhardwaj/vocult/files/test_video.mp4
Translated segments: 42
Hindi voice: hi-IN-SwaraNeural
Video duration: 138.00 seconds
Generated 1: मनुष्य के रूप में, हम एक दूसरे को समझते हैं।
Generated 2: समझ में आ गया.


CancelledError: 

In [9]:

import asyncio
import json
import re
import subprocess
from pathlib import Path

import edge_tts
import imageio_ffmpeg
import numpy as np
import soundfile as sf
import torch
import whisperx
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==================================================
# 1. CONFIGURATION: CHANGE ONLY THE VIDEO FILE
# ==================================================

BASE_DIR = Path("/Users/chaitanyabhardwaj/vocult/files")

# Set this to the video you want to translate.
# Example: "my_new_video.mp4"
VIDEO_NAME = "Chaitanya Video Assignment.mp4"
VIDEO_PATH = BASE_DIR / VIDEO_NAME

OUTPUT_DIR = BASE_DIR / "hindi_output" / VIDEO_PATH.stem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTED_AUDIO = OUTPUT_DIR / "original_audio.wav"
TRANSCRIPT_JSON = OUTPUT_DIR / "transcript.json"
TRANSLATED_JSON = OUTPUT_DIR / "translated_hindi.json"
HINDI_AUDIO = OUTPUT_DIR / "hindi_audio.wav"
FINAL_VIDEO = OUTPUT_DIR / f"{VIDEO_PATH.stem}_hindi.mp4"

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
SAMPLE_RATE = 24000
VOICE = "hi-IN-SwaraNeural"

WHISPER_MODEL = "small"
NLLB_MODEL = "facebook/nllb-200-distilled-600M"

# English input -> Hindi output
SOURCE_LANG = "eng_Latn"
TARGET_LANG = "hin_Deva"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

# ==================================================
# 2. CHECK INPUT
# ==================================================

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Video not found: {VIDEO_PATH}\n"
        f"Place the video inside {BASE_DIR} and update VIDEO_NAME."
    )

print("Input video:", VIDEO_PATH)
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR)

# ==================================================
# 3. EXTRACT AUDIO
# ==================================================

subprocess.run([
    FFMPEG, "-y",
    "-i", str(VIDEO_PATH),
    "-vn",
    "-ac", "1",
    "-ar", "16000",
    "-c:a", "pcm_s16le",
    str(EXTRACTED_AUDIO)
], check=True)

print("Original audio extracted.")

# ==================================================
# 4. TRANSCRIBE USING WHISPERX
# ==================================================

audio = whisperx.load_audio(str(EXTRACTED_AUDIO))

asr_model = whisperx.load_model(
    WHISPER_MODEL,
    device=DEVICE,
    compute_type=COMPUTE_TYPE
)

result = asr_model.transcribe(
    audio,
    batch_size=4 if DEVICE == "cuda" else 1
)

detected_language = result.get("language", "en")
print("Detected source language:", detected_language)

# This script uses English as the source for NLLB.
# For non-English source videos, see the note below.

segments = []

for i, item in enumerate(result.get("segments", [])):
    text = item.get("text", "").strip()
    if not text:
        continue

    segments.append({
        "id": i,
        "start": float(item["start"]),
        "end": float(item["end"]),
        "text": text
    })

if not segments:
    raise RuntimeError("No speech was detected in the video.")

with open(TRANSCRIPT_JSON, "w", encoding="utf-8") as f:
    json.dump(segments, f, ensure_ascii=False, indent=2)

print(f"Transcribed {len(segments)} speech segments.")

# Free up transcription model memory before translation.
del asr_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ==================================================
# 5. LOAD NLLB AND TRANSLATE INTO HINDI
# ==================================================

tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
translator = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL)
translator.to(DEVICE)
translator.eval()

tokenizer.src_lang = SOURCE_LANG
target_token_id = tokenizer.convert_tokens_to_ids(TARGET_LANG)


def translate_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        output = translator.generate(
            **inputs,
            forced_bos_token_id=target_token_id,
            max_new_tokens=256,
            num_beams=4
        )

    return tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )[0].strip()


translated_segments = []

for i, segment in enumerate(segments):
    hindi_text = translate_text(segment["text"])

    translated_segments.append({
        **segment,
        "translated_text": hindi_text
    })

    print(f"{i + 1}/{len(segments)}: {hindi_text}")

with open(TRANSLATED_JSON, "w", encoding="utf-8") as f:
    json.dump(
        translated_segments,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Hindi transcript saved:", TRANSLATED_JSON)

# Release translation model memory.
del translator
del tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ==================================================
# 6. GENERATE HINDI SPEECH
# ==================================================

async def generate_speech():
    generated = []

    for i, segment in enumerate(translated_segments):
        text = segment["translated_text"].strip()
        if not text:
            continue

        mp3_path = OUTPUT_DIR / f"segment_{i:04d}.mp3"
        wav_path = OUTPUT_DIR / f"segment_{i:04d}.wav"

        await edge_tts.Communicate(
            text=text,
            voice=VOICE
        ).save(str(mp3_path))

        subprocess.run([
            FFMPEG, "-y",
            "-i", str(mp3_path),
            "-ac", "1",
            "-ar", str(SAMPLE_RATE),
            "-c:a", "pcm_s16le",
            str(wav_path)
        ], check=True, capture_output=True)

        speech, sr = sf.read(wav_path, dtype="float32")

        if speech.ndim > 1:
            speech = speech.mean(axis=1)

        generated.append({
            "start": segment["start"],
            "end": segment["end"],
            "audio": speech
        })

        print(f"Generated Hindi speech: {i + 1}/{len(translated_segments)}")

    return generated


generated_segments = await generate_speech()

if not generated_segments:
    raise RuntimeError("Hindi speech generation failed.")

# ==================================================
# 7. ALIGN SPEECH WITH VIDEO TIMELINE
# ==================================================

probe = subprocess.run(
    [FFMPEG, "-i", str(VIDEO_PATH)],
    capture_output=True,
    text=True
)

match = re.search(
    r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)",
    probe.stderr
)

if not match:
    raise RuntimeError("Could not determine video duration.")

duration = (
    int(match.group(1)) * 3600
    + int(match.group(2)) * 60
    + float(match.group(3))
)

total_samples = int(duration * SAMPLE_RATE)
timeline = np.zeros(total_samples, dtype=np.float32)

for item in generated_segments:
    start = max(0, int(item["start"] * SAMPLE_RATE))
    speech = item["audio"]

    if start >= total_samples:
        continue

    # Avoid adding overlapping segments at the same samples.
    # This keeps the loudness under control, although it does
    # not perform full dialogue separation.
    end = min(start + len(speech), total_samples)
    speech = speech[:end - start]

    timeline[start:end] += speech

peak = np.max(np.abs(timeline)) if len(timeline) else 0
if peak > 0.99:
    timeline *= 0.99 / peak

sf.write(HINDI_AUDIO, timeline, SAMPLE_RATE)
print("Hindi audio saved:", HINDI_AUDIO)

# ==================================================
# 8. MERGE HINDI AUDIO WITH THE VIDEO
# ==================================================

subprocess.run([
    FFMPEG, "-y",
    "-i", str(VIDEO_PATH),
    "-i", str(HINDI_AUDIO),
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-c:v", "copy",
    "-c:a", "aac",
    "-b:a", "192k",
    "-t", str(duration),
    "-movflags", "+faststart",
    str(FINAL_VIDEO)
], check=True)

print("\nHindi dubbing completed!")
print("Final video:", FINAL_VIDEO)
print("Hindi transcript:", TRANSLATED_JSON)
print("Hindi audio:", HINDI_AUDIO)


Input video: /Users/chaitanyabhardwaj/vocult/files/Chaitanya Video Assignment.mp4
Device: cpu
Output directory: /Users/chaitanyabhardwaj/vocult/files/hindi_output/Chaitanya Video Assignment
Original audio extracted.


ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 13.1.6 (clang-1316.0.21.2.5)
  configuration: --prefix=/Volumes/tempdisk/sw --extra-cflags=-fno-stack-check --arch=arm64 --cc=/usr/bin/clang --enable-gpl --enable-libvmaf --enable-libopenjpeg --enable-libopus --enable-libmp3lame --enable-libx264 --enable-libx265 --enable-libvpx --enable-libwebp --enable-libass --enable-libfreetype --enable-fontconfig --enable-libtheora --enable-libvorbis --enable-libsnappy --enable-libaom --enable-libvidstab --enable-libzimg --enable-libsvtav1 --enable-libharfbuzz --enable-libkvazaar --pkg-config-flags=--static --enable-ffplay --enable-postproc --enable-neon --enable-runtime-cpudetect --disable-indev=qtkit --disable-indev=x11grab_xcb
  libavutil      59. 39.100 / 59. 39.100
  libavcodec     61. 19.100 / 61. 19.100
  libavformat    61.  7.100 / 61.  7.100
  libavdevice    61.  3.100 / 61.  3.100
  libavfilter    10.  4.100 / 10.  4.100
  libswscale      8. 

2026-10-22 18:40:01 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-10-22 18:40:01 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/whisperx/assets/pytorch_model.bin`


2026-10-22 18:40:02 - whisperx.asr - WARNING - Audio is shorter than 30s, language detection may be inaccurate
2026-10-22 18:40:03 - whisperx.asr - INFO - Detected language: en (0.75) in first 30s of audio
Detected source language: en
Transcribed 1 speech segments.
1/1: नमस्कार, मेरा नाम चेतन नेह भारद्वाज है और मैं वर्तमान में महाराजा सुरजमल इंस्टीट्यूट ऑफ टेक्नोलॉजी से इलेक्ट्रॉनिक्स और संचार इंजीनियरिंग में स्नातक की डिग्री प्राप्त कर रहा हूं। मेरे पास बिज़नेस एनालिस्ट इंटर्न के रूप में विप्रो में इंटर्नशिप का अनुभव है, जहां मैंने विभिन्न उपकरणों और तकनीक का उपयोग करके व्यावसायिक अंतर्दृष्टि बनाने पर काम किया है। मेरे शिक्षाविदों के अलावा, मैं दिल्ली मेट्रो के रोट्रैक क्लब का एक सक्रिय सदस्य हूं। मेरा शौक यात्रा करना है और मुझे नई जगहों और अनुभवों का पता लगाना पसंद है। धन्यवाद।
Hindi transcript saved: /Users/chaitanyabhardwaj/vocult/files/hindi_output/Chaitanya Video Assignment/translated_hindi.json
Generated Hindi speech: 1/1
Hindi audio saved: /Users/chaitanyabhardwaj/vocult/files/

ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 13.1.6 (clang-1316.0.21.2.5)
  configuration: --prefix=/Volumes/tempdisk/sw --extra-cflags=-fno-stack-check --arch=arm64 --cc=/usr/bin/clang --enable-gpl --enable-libvmaf --enable-libopenjpeg --enable-libopus --enable-libmp3lame --enable-libx264 --enable-libx265 --enable-libvpx --enable-libwebp --enable-libass --enable-libfreetype --enable-fontconfig --enable-libtheora --enable-libvorbis --enable-libsnappy --enable-libaom --enable-libvidstab --enable-libzimg --enable-libsvtav1 --enable-libharfbuzz --enable-libkvazaar --pkg-config-flags=--static --enable-ffplay --enable-postproc --enable-neon --enable-runtime-cpudetect --disable-indev=qtkit --disable-indev=x11grab_xcb
  libavutil      59. 39.100 / 59. 39.100
  libavcodec     61. 19.100 / 61. 19.100
  libavformat    61.  7.100 / 61.  7.100
  libavdevice    61.  3.100 / 61.  3.100
  libavfilter    10.  4.100 / 10.  4.100
  libswscale      8. 


Hindi dubbing completed!
Final video: /Users/chaitanyabhardwaj/vocult/files/hindi_output/Chaitanya Video Assignment/Chaitanya Video Assignment_hindi.mp4
Hindi transcript: /Users/chaitanyabhardwaj/vocult/files/hindi_output/Chaitanya Video Assignment/translated_hindi.json
Hindi audio: /Users/chaitanyabhardwaj/vocult/files/hindi_output/Chaitanya Video Assignment/hindi_audio.wav


[mp4 @ 0x12d00eaa0] Starting second pass: moving the moov atom to the beginning of the file
[out#0/mp4 @ 0x6000011380c0] video:7562KiB audio:327KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.204063%
frame=  808 fps=0.0 q=-1.0 Lsize=    7905KiB time=00:00:26.93 bitrate=2404.4kbits/s speed= 109x    
[aac @ 0x12d0101c0] Qavg: 65509.516


In [1]:
from IPython.display import display
import ipywidgets as widgets
from pathlib import Path

BASE_DIR = Path("/Users/chaitanyabhardwaj/vocult/files")
BASE_DIR.mkdir(parents=True, exist_ok=True)

upload = widgets.FileUpload(
    accept=".mp4,.mov,.mkv,.avi",
    multiple=False
)

display(upload)

FileUpload(value=(), accept='.mp4,.mov,.mkv,.avi', description='Upload')

In [9]:

import asyncio
import json
import re
import subprocess
from pathlib import Path

import edge_tts
import imageio_ffmpeg
import numpy as np
import soundfile as sf
import torch
import whisperx

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from IPython.display import display
import ipywidgets as widgets

# =========================================================
# 1. CONFIGURATION
# =========================================================

BASE_DIR = Path.home() / "vocult" / "files"

# Change this only if your Vocult folder is in another location.
if not BASE_DIR.exists():
    BASE_DIR = Path("/Users/chaitanyabhardwaj/vocult/files")

OUTPUT_ROOT = BASE_DIR / "hindi_output"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VIDEO_EXTENSIONS = {
    ".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"
}

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

WHISPER_MODEL = "small"
NLLB_MODEL = "facebook/nllb-200-distilled-600M"

VOICE = "hi-IN-SwaraNeural"
SAMPLE_RATE = 24000

# This version assumes the source speech is English.
SOURCE_LANG = "eng_Latn"
TARGET_LANG = "hin_Deva"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

# =========================================================
# 2. SELECT A VIDEO AUTOMATICALLY
# =========================================================

if not BASE_DIR.exists():
    raise FileNotFoundError(
        f"Vocult folder does not exist: {BASE_DIR}\n"
        "Update BASE_DIR to your actual files folder."
    )

video_files = sorted(
    [
        p for p in BASE_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower() in VIDEO_EXTENSIONS
    ],
    key=lambda p: p.name.lower()
)

if not video_files:
    raise FileNotFoundError(
        f"No supported video files found in {BASE_DIR}.\n"
        "Copy your video into this folder and rerun this cell."
    )

print("Available videos:")
for i, path in enumerate(video_files):
    print(f"{i}: {path.name}")

# Interactive selector: choose any available video.
video_selector = widgets.Dropdown(
    options=[
        (p.name, str(p)) for p in video_files
    ],
    description="Video:",
    layout=widgets.Layout(width="85%")
)

run_button = widgets.Button(
    description="Translate to Hindi",
    button_style="success",
    icon="play"
)

display(video_selector, run_button)

# =========================================================
# 3. VIDEO DURATION HELPER
# =========================================================

def get_duration(path):
    result = subprocess.run(
        [
            FFMPEG, "-i", str(path),
            "-f", "null", "-"
        ],
        capture_output=True,
        text=True
    )

    match = re.search(
        r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)",
        result.stderr
    )

    if not match:
        raise RuntimeError(
            f"Could not read duration for {path.name}"
        )

    h, m, s = match.groups()
    return int(h) * 3600 + int(m) * 60 + float(s)


# =========================================================
# 4. MAIN VIDEO TRANSLATION PIPELINE
# =========================================================

def translate_video(video_path):

    video_path = Path(video_path)

    if not video_path.exists():
        raise FileNotFoundError(video_path)

    # Each video gets its own output folder.
    output_dir = OUTPUT_ROOT / video_path.stem
    output_dir.mkdir(parents=True, exist_ok=True)

    original_audio = output_dir / "original_audio.wav"
    transcript_json = output_dir / "transcript.json"
    translated_json = output_dir / "translated_hindi.json"
    hindi_audio_path = output_dir / "hindi_audio.wav"
    final_video = output_dir / f"{video_path.stem}_hindi.mp4"

    print("\nSelected:", video_path.name)
    print("Output:", output_dir)
    print("Device:", DEVICE)

    # -----------------------------------------------------
    # A. Extract original audio
    # -----------------------------------------------------

    subprocess.run(
        [
            FFMPEG, "-y",
            "-i", str(video_path),
            "-vn",
            "-ac", "1",
            "-ar", "16000",
            "-c:a", "pcm_s16le",
            str(original_audio)
        ],
        check=True,
        capture_output=True
    )

    print("1/6: Original audio extracted.")

    duration = get_duration(video_path)

    # -----------------------------------------------------
    # B. Transcribe speech with WhisperX
    # -----------------------------------------------------

    audio = whisperx.load_audio(str(original_audio))

    asr_model = whisperx.load_model(
        WHISPER_MODEL,
        device=DEVICE,
        compute_type=COMPUTE_TYPE
    )

    result = asr_model.transcribe(
        audio,
        batch_size=4 if DEVICE == "cuda" else 1
    )

    detected_language = result.get("language", "en")
    print("Detected language:", detected_language)

    # The translation configuration below expects English.
    if detected_language not in ("en", "eng"):
        del asr_model
        raise ValueError(
            f"Detected source language: {detected_language}. "
            "This pipeline currently translates English into Hindi. "
            "Configure the NLLB source language for this input language."
        )

    segments = []

    for i, item in enumerate(result.get("segments", [])):
        text = item.get("text", "").strip()

        if not text:
            continue

        segments.append({
            "id": i,
            "start": float(item["start"]),
            "end": float(item["end"]),
            "text": text
        })

    del asr_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if not segments:
        raise RuntimeError("No speech detected.")

    with open(transcript_json, "w", encoding="utf-8") as f:
        json.dump(segments, f, ensure_ascii=False, indent=2)

    print(f"2/6: Transcribed {len(segments)} segments.")

    # -----------------------------------------------------
    # C. Translate text into Hindi
    # -----------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
    translator = AutoModelForSeq2SeqLM.from_pretrained(
        NLLB_MODEL
    ).to(DEVICE)

    tokenizer.src_lang = SOURCE_LANG
    target_id = tokenizer.convert_tokens_to_ids(TARGET_LANG)

    translated_segments = []

    def translate_text(text):
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(DEVICE)

        with torch.no_grad():
            output = translator.generate(
                **inputs,
                forced_bos_token_id=target_id,
                max_new_tokens=256,
                num_beams=4
            )

        return tokenizer.batch_decode(
            output, skip_special_tokens=True
        )[0].strip()

    for i, seg in enumerate(segments):
        translated = translate_text(seg["text"])

        translated_segments.append({
            **seg,
            "translated_text": translated
        })

        print(
            f"Translated {i + 1}/{len(segments)}: "
            f"{translated[:80]}"
        )

    with open(translated_json, "w", encoding="utf-8") as f:
        json.dump(
            translated_segments,
            f,
            ensure_ascii=False,
            indent=2
        )

    del translator, tokenizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("3/6: Hindi translation saved.")

    # -----------------------------------------------------
    # D. Generate Hindi TTS segment files
    # -----------------------------------------------------

    async def generate_segments():
        generated = []

        for i, seg in enumerate(translated_segments):
            text = seg["translated_text"].strip()

            if not text:
                continue

            mp3_path = output_dir / f"speech_{i:04d}.mp3"
            wav_path = output_dir / f"speech_{i:04d}.wav"

            await edge_tts.Communicate(
                text=text,
                voice=VOICE
            ).save(str(mp3_path))

            subprocess.run(
                [
                    FFMPEG, "-y",
                    "-i", str(mp3_path),
                    "-ac", "1",
                    "-ar", str(SAMPLE_RATE),
                    "-c:a", "pcm_s16le",
                    str(wav_path)
                ],
                check=True,
                capture_output=True
            )

            speech, _ = sf.read(
                wav_path, dtype="float32"
            )

            if speech.ndim > 1:
                speech = speech.mean(axis=1)

            generated.append({
                "start": seg["start"],
                "end": seg["end"],
                "audio": speech
            })

            print(
                f"Speech {i + 1}/{len(translated_segments)}"
            )

        return generated

    generated = asyncio.run(generate_segments())

    if not generated:
        raise RuntimeError("No Hindi speech generated.")

    print("4/6: Hindi speech generated.")

    # -----------------------------------------------------
    # E. Place speech on the video timeline
    # -----------------------------------------------------

    total_samples = int(np.ceil(duration * SAMPLE_RATE))
    timeline = np.zeros(total_samples, dtype=np.float32)

    # Limit the peak amplitude of individual segments.
    for item in generated:
        start = max(
            0, int(item["start"] * SAMPLE_RATE)
        )
        speech = item["audio"]

        if start >= total_samples:
            continue

        end = min(start + len(speech), total_samples)
        speech = speech[:end - start]

        # Apply a short fade at segment boundaries to reduce clicks.
        fade = min(
            int(0.008 * SAMPLE_RATE),
            len(speech) // 2
        )

        speech = speech.copy()

        if fade > 0:
            speech[:fade] *= np.linspace(
                0, 1, fade, dtype=np.float32
            )
            speech[-fade:] *= np.linspace(
                1, 0, fade, dtype=np.float32
            )

        # Mix overlapping speech without hard clipping.
        timeline[start:end] += speech

    peak = (
        float(np.max(np.abs(timeline)))
        if len(timeline) else 0
    )

    if peak > 0.98:
        timeline *= 0.98 / peak

    sf.write(
        hindi_audio_path,
        timeline,
        SAMPLE_RATE
    )

    print("5/6: Hindi audio aligned.")

    # -----------------------------------------------------
    # F. Merge video and Hindi audio
    # -----------------------------------------------------

    subprocess.run(
        [
            FFMPEG, "-y",
            "-i", str(video_path),
            "-i", str(hindi_audio_path),
            "-map", "0:v:0",
            "-map", "1:a:0",
            "-c:v", "copy",
            "-c:a", "aac",
            "-b:a", "192k",
            "-t", str(duration),
            "-movflags", "+faststart",
            str(final_video)
        ],
        check=True
    )

    print("\n6/6: Hindi dubbing completed!")
    print("Final video:", final_video)
    print("Translated JSON:", translated_json)
    print("Hindi audio:", hindi_audio_path)

    return final_video


# =========================================================
# 5. RUN WHEN THE BUTTON IS CLICKED
# =========================================================

def on_translate_clicked(button):
    button.disabled = True

    try:
        selected_video = Path(video_selector.value)
        result_path = translate_video(selected_video)
        print("\nSUCCESS:", result_path)
    except Exception as exc:
        print(f"\nERROR: {type(exc).__name__}: {exc}")
        raise
    finally:
        button.disabled = False


run_button.on_click(on_translate_clicked)



Available videos:
0: Chaitanya Video Assignment.mp4
1: test.mp4
2: test_video.mp4


Dropdown(description='Video:', layout=Layout(width='85%'), options=(('Chaitanya Video Assignment.mp4', '/Users…

Button(button_style='success', description='Translate to Hindi', icon='play', style=ButtonStyle())

In [12]:

VIDEO_PATH = video_files[2]

print("Selected video:", VIDEO_PATH)
print("File exists:", VIDEO_PATH.exists())


Selected video: /Users/chaitanyabhardwaj/vocult/files/test_video.mp4
File exists: True


In [16]:

import subprocess
import imageio_ffmpeg
from pathlib import Path

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

def inspect_video(path):
    result = subprocess.run(
        [FFMPEG, "-hide_banner", "-i", str(path)],
        capture_output=True,
        text=True
    )

    print(f"\nFile: {path.name}")

    streams = [
        line.strip()
        for line in result.stderr.splitlines()
        if "Stream #" in line
    ]

    if streams:
        for stream in streams:
            print(stream)
    else:
        print("No streams detected.")

for path in video_files:
    inspect_video(path)



File: Chaitanya Video Assignment.mp4
Stream #0:0[0x1](und): Video: h264 (Constrained Baseline) (avc1 / 0x31637661), yuv420p(tv, bt709, progressive), 1620x1080, 2300 kb/s, 30 fps, 30 tbr, 15360 tbn (default)
Stream #0:1[0x2](und): Audio: aac (LC) (mp4a / 0x6134706D), 44100 Hz, mono, fltp, 128 kb/s (default)

File: test.mp4
Stream #0:0[0x1](und): Video: h264 (High) (avc1 / 0x31637661), yuv420p(tv, bt709, progressive), 1920x1080 [SAR 1:1 DAR 16:9], 1849 kb/s, 30 fps, 30 tbr, 15360 tbn (default)
Stream #0:1[0x2](und): Audio: aac (LC) (mp4a / 0x6134706D), 44100 Hz, stereo, fltp, 128 kb/s (default)

File: test_video.mp4
Stream #0:0[0x1](und): Video: h264 (High) (avc1 / 0x31637661), yuv420p(tv, bt709, progressive), 1920x1080 [SAR 1:1 DAR 16:9], 1849 kb/s, 30 fps, 30 tbr, 15360 tbn (default)


In [ ]:

import asyncio
import json
import re
import subprocess
from pathlib import Path

import edge_tts
import imageio_ffmpeg
import numpy as np
import soundfile as sf
import torch
import whisperx
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==================================================
# 1. CONFIGURATION: CHANGE ONLY THE VIDEO FILE
# ==================================================

BASE_DIR = Path("/Users/chaitanyabhardwaj/vocult/files")

# Set this to the video you want to translate.
# Example: "my_new_video.mp4"
VIDEO_NAME = "upload.mp4"
VIDEO_PATH = BASE_DIR / VIDEO_NAME

OUTPUT_DIR = BASE_DIR / "hindi_output" / VIDEO_PATH.stem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTED_AUDIO = OUTPUT_DIR / "original_audio.wav"
TRANSCRIPT_JSON = OUTPUT_DIR / "transcript.json"
TRANSLATED_JSON = OUTPUT_DIR / "translated_hindi.json"
HINDI_AUDIO = OUTPUT_DIR / "hindi_audio.wav"
FINAL_VIDEO = OUTPUT_DIR / f"{VIDEO_PATH.stem}_hindi.mp4"

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
SAMPLE_RATE = 24000
VOICE = "hi-IN-SwaraNeural"

WHISPER_MODEL = "small"
NLLB_MODEL = "facebook/nllb-200-distilled-600M"

# English input -> Hindi output
SOURCE_LANG = "eng_Latn"
TARGET_LANG = "hin_Deva"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

# ==================================================
# 2. CHECK INPUT
# ==================================================

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Video not found: {VIDEO_PATH}\n"
        f"Place the video inside {BASE_DIR} and update VIDEO_NAME."
    )

print("Input video:", VIDEO_PATH)
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR)

# ==================================================
# 3. EXTRACT AUDIO
# ==================================================

subprocess.run([
    FFMPEG, "-y",
    "-i", str(VIDEO_PATH),
    "-vn",
    "-ac", "1",
    "-ar", "16000",
    "-c:a", "pcm_s16le",
    str(EXTRACTED_AUDIO)
], check=True)

print("Original audio extracted.")

# ==================================================
# 4. TRANSCRIBE USING WHISPERX
# ==================================================

audio = whisperx.load_audio(str(EXTRACTED_AUDIO))

asr_model = whisperx.load_model(
    WHISPER_MODEL,
    device=DEVICE,
    compute_type=COMPUTE_TYPE
)

result = asr_model.transcribe(
    audio,
    batch_size=4 if DEVICE == "cuda" else 1
)

detected_language = result.get("language", "en")
print("Detected source language:", detected_language)

# This script uses English as the source for NLLB.
# For non-English source videos, see the note below.

segments = []

for i, item in enumerate(result.get("segments", [])):
    text = item.get("text", "").strip()
    if not text:
        continue

    segments.append({
        "id": i,
        "start": float(item["start"]),
        "end": float(item["end"]),
        "text": text
    })

if not segments:
    raise RuntimeError("No speech was detected in the video.")

with open(TRANSCRIPT_JSON, "w", encoding="utf-8") as f:
    json.dump(segments, f, ensure_ascii=False, indent=2)

print(f"Transcribed {len(segments)} speech segments.")

# Free up transcription model memory before translation.
del asr_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ==================================================
# 5. LOAD NLLB AND TRANSLATE INTO HINDI
# ==================================================

tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
translator = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL)
translator.to(DEVICE)
translator.eval()

tokenizer.src_lang = SOURCE_LANG
target_token_id = tokenizer.convert_tokens_to_ids(TARGET_LANG)


def translate_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        output = translator.generate(
            **inputs,
            forced_bos_token_id=target_token_id,
            max_new_tokens=256,
            num_beams=4
        )

    return tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )[0].strip()


translated_segments = []

for i, segment in enumerate(segments):
    hindi_text = translate_text(segment["text"])

    translated_segments.append({
        **segment,
        "translated_text": hindi_text
    })

    print(f"{i + 1}/{len(segments)}: {hindi_text}")

with open(TRANSLATED_JSON, "w", encoding="utf-8") as f:
    json.dump(
        translated_segments,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Hindi transcript saved:", TRANSLATED_JSON)

# Release translation model memory.
del translator
del tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ==================================================
# 6. GENERATE HINDI SPEECH
# ==================================================

async def generate_speech():
    generated = []

    for i, segment in enumerate(translated_segments):
        text = segment["translated_text"].strip()
        if not text:
            continue

        mp3_path = OUTPUT_DIR / f"segment_{i:04d}.mp3"
        wav_path = OUTPUT_DIR / f"segment_{i:04d}.wav"

        await edge_tts.Communicate(
            text=text,
            voice=VOICE
        ).save(str(mp3_path))

        subprocess.run([
            FFMPEG, "-y",
            "-i", str(mp3_path),
            "-ac", "1",
            "-ar", str(SAMPLE_RATE),
            "-c:a", "pcm_s16le",
            str(wav_path)
        ], check=True, capture_output=True)

        speech, sr = sf.read(wav_path, dtype="float32")

        if speech.ndim > 1:
            speech = speech.mean(axis=1)

        generated.append({
            "start": segment["start"],
            "end": segment["end"],
            "audio": speech
        })

        print(f"Generated Hindi speech: {i + 1}/{len(translated_segments)}")

    return generated


generated_segments = await generate_speech()

if not generated_segments:
    raise RuntimeError("Hindi speech generation failed.")

# ==================================================
# 7. ALIGN SPEECH WITH VIDEO TIMELINE
# ==================================================

probe = subprocess.run(
    [FFMPEG, "-i", str(VIDEO_PATH)],
    capture_output=True,
    text=True
)

match = re.search(
    r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)",
    probe.stderr
)

if not match:
    raise RuntimeError("Could not determine video duration.")

duration = (
    int(match.group(1)) * 3600
    + int(match.group(2)) * 60
    + float(match.group(3))
)

total_samples = int(duration * SAMPLE_RATE)
timeline = np.zeros(total_samples, dtype=np.float32)

for item in generated_segments:
    start = max(0, int(item["start"] * SAMPLE_RATE))
    speech = item["audio"]

    if start >= total_samples:
        continue

    # Avoid adding overlapping segments at the same samples.
    # This keeps the loudness under control, although it does
    # not perform full dialogue separation.
    end = min(start + len(speech), total_samples)
    speech = speech[:end - start]

    timeline[start:end] += speech

peak = np.max(np.abs(timeline)) if len(timeline) else 0
if peak > 0.99:
    timeline *= 0.99 / peak

sf.write(HINDI_AUDIO, timeline, SAMPLE_RATE)
print("Hindi audio saved:", HINDI_AUDIO)

# ==================================================
# 8. MERGE HINDI AUDIO WITH THE VIDEO
# ==================================================

subprocess.run([
    FFMPEG, "-y",
    "-i", str(VIDEO_PATH),
    "-i", str(HINDI_AUDIO),
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-c:v", "copy",
    "-c:a", "aac",
    "-b:a", "192k",
    "-t", str(duration),
    "-movflags", "+faststart",
    str(FINAL_VIDEO)
], check=True)

print("\nHindi dubbing completed!")
print("Final video:", FINAL_VIDEO)
print("Hindi transcript:", TRANSLATED_JSON)
print("Hindi audio:", HINDI_AUDIO)


In [2]:
import ipywidgets as widgets
from IPython.display import display

display(widgets.FileUpload(description="Choose Video"))

FileUpload(value=(), description='Choose Video')

In [1]:

from pathlib import Path
import inspect
import traceback
import subprocess
import sys
import os

import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink, HTML
import imageio_ffmpeg

# ---------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ---------------------------------------------------------

BASE_DIR = Path.home() / "vocult" / "files"

UPLOAD_DIR = BASE_DIR / "user_uploads"
OUTPUT_DIR = BASE_DIR / "translated_videos"

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FFMPEG_PATH = imageio_ffmpeg.get_ffmpeg_exe()

# Supported languages shown in the dropdown.
# The pipeline must support the selected language.
LANGUAGES = {
    "Hindi": "hi",
    "English": "en",
    "Spanish": "es",
    "French": "fr",
    "German": "de",
    "Italian": "it",
    "Portuguese": "pt",
    "Russian": "ru",
    "Japanese": "ja",
    "Chinese (Simplified)": "zh",
    "Arabic": "ar",
    "Bengali": "bn",
    "Tamil": "ta",
    "Telugu": "te",
    "Punjabi": "pa",
    "Urdu": "ur",
}

# ---------------------------------------------------------
# 2. USER INTERFACE
# ---------------------------------------------------------

video_upload = widgets.FileUpload(
    accept=".mp4,.mov,.mkv,.avi,.webm",
    multiple=False,
    description="Choose Video",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%")
)

language_dropdown = widgets.Dropdown(
    options=list(LANGUAGES.keys()),
    value="Hindi",
    description="Translate to:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%")
)

translate_button = widgets.Button(
    description="Translate Video",
    icon="language",
    button_style="success",
    tooltip="Translate the uploaded video",
    layout=widgets.Layout(width="100%", height="42px")
)

status_output = widgets.Output()
progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Progress:",
    bar_style="info",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%")
)

download_output = widgets.Output()

# ---------------------------------------------------------
# 3. HANDLE IPYWIDGETS 7 AND 8 UPLOAD FORMATS
# ---------------------------------------------------------

def get_uploaded_file(upload_widget):
    """
    Returns (filename, file_bytes) for ipywidgets 7 or 8.
    """

    value = upload_widget.value

    if not value:
        raise ValueError("Please choose a video file first.")

    # ipywidgets 7 may return a dictionary.
    if isinstance(value, dict):
        filename, info = next(iter(value.items()))

        if isinstance(info, dict):
            content = info["content"]
        else:
            content = info.content

    # ipywidgets 8 normally returns a tuple of file records.
    elif isinstance(value, (tuple, list)):
        info = value[0]

        if isinstance(info, dict):
            filename = info["name"]
            content = info["content"]
        else:
            filename = info.name
            content = info.content

    else:
        raise ValueError(
            "Unsupported upload format. Update ipywidgets and restart the kernel."
        )

    return Path(filename).name, bytes(content)


# ---------------------------------------------------------
# 4. SAVE THE VIDEO
# ---------------------------------------------------------

def save_uploaded_video():
    filename, content = get_uploaded_file(video_upload)

    allowed_extensions = {
        ".mp4", ".mov", ".mkv", ".avi", ".webm"
    }

    extension = Path(filename).suffix.lower()

    if extension not in allowed_extensions:
        raise ValueError(f"Unsupported video format: {extension}")

    input_path = UPLOAD_DIR / filename
    input_path.write_bytes(content)

    if not input_path.exists() or input_path.stat().st_size == 0:
        raise RuntimeError("The uploaded video could not be saved.")

    return input_path


# ---------------------------------------------------------
# 5. VALIDATE VIDEO WITH FFMPEG
# ---------------------------------------------------------

def validate_video(input_path):
    """
    Check that FFmpeg can read the uploaded video.
    """

    command = [
        FFMPEG_PATH,
        "-hide_banner",
        "-i", str(input_path),
        "-f", "null",
        "-"
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    # FFmpeg can return a non-zero status for a null output
    # depending on the input. Check whether it recognizes a stream.
    diagnostic = result.stderr.lower()

    if "input #" not in diagnostic or "video:" not in diagnostic:
        raise RuntimeError(
            "FFmpeg could not validate the video. "
            "Check the file format and whether the file is corrupted.\n"
            + result.stderr[-2500:]
        )


# ---------------------------------------------------------
# 6. CONNECT TO YOUR VOCULT PIPELINE
# ---------------------------------------------------------

def run_vocult_pipeline(input_path, target_language, output_path):
    """
    Calls the existing Vocult translation pipeline.

    Expected pipeline signature:
        translate_video(input_path, target_language, output_path)

    Update your existing translate_video() function to accept
    these three arguments and use target_language throughout
    transcription, translation, speech generation and merging.
    """

    if "translate_video" not in globals():
        raise NameError(
            "translate_video() is not defined. "
            "Run your Vocult pipeline definition cell first."
        )

    pipeline = globals()["translate_video"]
    parameters = inspect.signature(pipeline).parameters

    # Pass only arguments the pipeline accepts.
    available_args = {
        "input_path": input_path,
        "video_path": input_path,
        "target_language": target_language,
        "language": target_language,
        "output_path": output_path,
        "output_video_path": output_path,
    }

    kwargs = {
        name: available_args[name]
        for name in parameters
        if name in available_args
    }

    required = [
        name for name, param in parameters.items()
        if param.default is inspect.Parameter.empty
        and param.kind in (
            inspect.Parameter.POSITIONAL_OR_KEYWORD,
            inspect.Parameter.KEYWORD_ONLY,
        )
    ]

    missing = [name for name in required if name not in kwargs]

    if missing:
        raise TypeError(
            "Your translate_video() function requires additional "
            f"arguments: {missing}. Adapt run_vocult_pipeline() "
            "to match its signature."
        )

    result = pipeline(**kwargs)

    # Accept pipelines that return the output path or save to output_path.
    if isinstance(result, (str, Path)):
        result_path = Path(result)
    else:
        result_path = output_path

    if not result_path.is_absolute():
        result_path = Path.cwd() / result_path

    if not result_path.exists():
        # Check the requested output location too.
        if output_path.exists():
            result_path = output_path
        else:
            raise FileNotFoundError(
                "The pipeline finished without creating its output video. "
                "Ensure translate_video() saves the final MP4."
            )

    return result_path


# ---------------------------------------------------------
# 7. BUTTON CLICK HANDLER
# ---------------------------------------------------------

def on_translate_clicked(_):
    translate_button.disabled = True
    download_output.clear_output()
    status_output.clear_output()

    progress.value = 0
    progress.bar_style = "info"

    try:
        with status_output:
            print("Step 1/4: Saving uploaded video...")

        input_path = save_uploaded_video()
        progress.value = 15

        with status_output:
            print(f"Input video: {input_path.name}")
            print("Step 2/4: Checking video with FFmpeg...")

        validate_video(input_path)
        progress.value = 25

        selected_language = LANGUAGES[language_dropdown.value]
        output_path = (
            OUTPUT_DIR
            / f"{input_path.stem}_{selected_language}.mp4"
        )

        with status_output:
            print(f"Target language: {language_dropdown.label}")
            print("Step 3/4: Running Vocult translation pipeline...")
            print(
                "This may take several minutes depending on "
                "video length and available hardware."
            )

        # Run your existing WhisperX + translation + TTS + FFmpeg pipeline.
        result_path = run_vocult_pipeline(
            input_path=input_path,
            target_language=selected_language,
            output_path=output_path,
        )

        progress.value = 100
        progress.bar_style = "success"

        with status_output:
            print("Step 4/4: Translation completed!")
            print(f"Output: {result_path}")

        with download_output:
            display(HTML("<h4>Translated video ready</h4>"))
            display(FileLink(str(result_path)))

    except Exception as error:
        progress.bar_style = "danger"

        with status_output:
            print(f"Translation failed: {error}")
            traceback.print_exc()

    finally:
        translate_button.disabled = False


translate_button.on_click(on_translate_clicked)

# ---------------------------------------------------------
# 8. DISPLAY THE INTERFACE
# ---------------------------------------------------------

app = widgets.VBox(
    [
        widgets.HTML(
            """
            <div style="padding:16px 18px;
                        border-radius:12px;
                        background:#172554;
                        color:white;">
                <h2 style="margin:0 0 8px 0;">
                    Vocult — AI Video Translator
                </h2>
                <p style="margin:0;">
                    Upload a video, select a language and generate
                    a translated version.
                </p>
            </div>
            """
        ),
        widgets.HTML("<h4>1. Upload your video</h4>"),
        video_upload,
        widgets.HTML("<h4>2. Select target language</h4>"),
        language_dropdown,
        translate_button,
        progress,
        status_output,
        download_output,
    ],
    layout=widgets.Layout(
        width="100%",
        max_width="800px",
        gap="12px",
        padding="12px"
    )
)

display(app)


In [ ]:

import asyncio
import json
import re
import subprocess
from pathlib import Path

import edge_tts
import imageio_ffmpeg
import numpy as np
import soundfile as sf
import torch
import whisperx
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==================================================
# 1. CONFIGURATION
# ==================================================

BASE_DIR = Path.home() / "vocult" / "files"

OUTPUT_ROOT = BASE_DIR / "translated_videos"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

SAMPLE_RATE = 24000
WHISPER_MODEL = "small"
NLLB_MODEL = "facebook/nllb-200-distilled-600M"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

# Language name, NLLB language code, Edge TTS voice
LANGUAGES = {
    "1": ("Hindi", "hin_Deva", "hi-IN-SwaraNeural"),
    "2": ("English", "eng_Latn", "en-US-AriaNeural"),
    "3": ("Spanish", "spa_Latn", "es-ES-ElviraNeural"),
    "4": ("French", "fra_Latn", "fr-FR-DeniseNeural"),
    "5": ("German", "deu_Latn", "de-DE-KatjaNeural"),
    "6": ("Japanese", "jpn_Jpan", "ja-JP-NanamiNeural"),
    "7": ("Chinese (Simplified)", "zho_Hans", "zh-CN-XiaoxiaoNeural"),
    "8": ("Arabic", "arb_Arab", "ar-SA-ZariyahNeural"),
    "9": ("Bengali", "ben_Beng", "bn-IN-TanishaaNeural"),
    "10": ("Tamil", "tam_Taml", "ta-IN-PallaviNeural"),
    "11": ("Telugu", "tel_Telu", "te-IN-ShrutiNeural"),
    "12": ("Punjabi", "pan_Guru", "pa-IN-VaaniNeural"),
    "13": ("Urdu", "urd_Arab", "ur-PK-UzmaNeural"),
}

SUPPORTED_EXTENSIONS = {
    ".mp4", ".mov", ".mkv", ".avi", ".webm"
}

# ==================================================
# 2. USER INPUT: VIDEO AND TARGET LANGUAGE
# ==================================================

print("=" * 55)
print("VOCULT — AI VIDEO TRANSLATOR")
print("=" * 55)
print("Video directory:", BASE_DIR)
print("Device:", DEVICE)

video_filename = input(
    "\nEnter video filename (e.g., test_video.mp4): "
).strip().strip('"').strip("'")

if not video_filename:
    raise ValueError("Please enter a video filename.")

if Path(video_filename).name != video_filename:
    raise ValueError(
        "Enter only the filename, not its full path. "
        "Example: test_video.mp4"
    )

VIDEO_PATH = BASE_DIR / video_filename

if not VIDEO_PATH.is_file():
    available = [
        p.name for p in BASE_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

    raise FileNotFoundError(
        f"Video not found: {VIDEO_PATH}\n"
        f"Available video files: {available}"
    )

if VIDEO_PATH.suffix.lower() not in SUPPORTED_EXTENSIONS:
    raise ValueError("Unsupported video format.")

print("\nSelect the target language:\n")

for number, (name, nllb_code, voice) in LANGUAGES.items():
    print(f"{number}. {name}")

choice = input("\nEnter language number: ").strip()

if choice not in LANGUAGES:
    raise ValueError("Invalid language selection.")

LANGUAGE_NAME, TARGET_LANG, VOICE = LANGUAGES[choice]

# WhisperX returns language codes such as "en" or "hi".
# Map common codes to NLLB's language codes.
NLLB_SOURCE_CODES = {
    "en": "eng_Latn",
    "hi": "hin_Deva",
    "es": "spa_Latn",
    "fr": "fra_Latn",
    "de": "deu_Latn",
    "ja": "jpn_Jpan",
    "zh": "zho_Hans",
    "ar": "arb_Arab",
    "bn": "ben_Beng",
    "ta": "tam_Taml",
    "te": "tel_Telu",
    "pa": "pan_Guru",
    "ur": "urd_Arab",
    "it": "ita_Latn",
    "pt": "por_Latn",
    "ru": "rus_Cyrl",
}

# Create a unique folder for this input and target language.
OUTPUT_DIR = OUTPUT_ROOT / f"{VIDEO_PATH.stem}_{TARGET_LANG}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTED_AUDIO = OUTPUT_DIR / "original_audio.wav"
TRANSCRIPT_JSON = OUTPUT_DIR / "transcript.json"
TRANSLATED_JSON = OUTPUT_DIR / "translated_transcript.json"
DUBBED_AUDIO = OUTPUT_DIR / "translated_audio.wav"
FINAL_VIDEO = OUTPUT_DIR / f"{VIDEO_PATH.stem}_{TARGET_LANG}.mp4"

print("\n" + "=" * 55)
print("TRANSLATION SETTINGS")
print("=" * 55)
print("Input video :", VIDEO_PATH)
print("Target      :", LANGUAGE_NAME)
print("NLLB code   :", TARGET_LANG)
print("TTS voice   :", VOICE)
print("Output      :", FINAL_VIDEO)
print("=" * 55)

# ==================================================
# 3. EXTRACT AUDIO
# ==================================================

subprocess.run(
    [
        FFMPEG, "-y",
        "-i", str(VIDEO_PATH),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        str(EXTRACTED_AUDIO),
    ],
    check=True,
)

print("\nOriginal audio extracted.")

# ==================================================
# 4. TRANSCRIBE USING WHISPERX
# ==================================================

audio = whisperx.load_audio(str(EXTRACTED_AUDIO))

asr_model = whisperx.load_model(
    WHISPER_MODEL,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)

result = asr_model.transcribe(
    audio,
    batch_size=4 if DEVICE == "cuda" else 1,
)

detected_language = result.get("language", "en")
print("Detected source language:", detected_language)

SOURCE_LANG = NLLB_SOURCE_CODES.get(detected_language)

if SOURCE_LANG is None:
    raise ValueError(
        f"No NLLB source-language mapping for '{detected_language}'. "
        "Add the correct NLLB language code before continuing."
    )

if SOURCE_LANG == TARGET_LANG:
    print(
        "Source and target languages are the same. "
        "The transcript will still be processed through the pipeline."
    )

segments = []

for i, item in enumerate(result.get("segments", [])):
    text = item.get("text", "").strip()

    if not text:
        continue

    segments.append(
        {
            "id": i,
            "start": float(item["start"]),
            "end": float(item["end"]),
            "text": text,
        }
    )

if not segments:
    raise RuntimeError("WhisperX did not detect any speech.")

with open(TRANSCRIPT_JSON, "w", encoding="utf-8") as f:
    json.dump(segments, f, ensure_ascii=False, indent=2)

print(f"Transcribed {len(segments)} speech segments.")

del asr_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ==================================================
# 5. LOAD NLLB AND TRANSLATE
# ==================================================

print("\nLoading translation model...")

tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
translator = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL)

translator.to(DEVICE)
translator.eval()

tokenizer.src_lang = SOURCE_LANG
target_token_id = tokenizer.convert_tokens_to_ids(TARGET_LANG)


def translate_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(DEVICE)

    with torch.no_grad():
        output = translator.generate(
            **inputs,
            forced_bos_token_id=target_token_id,
            max_new_tokens=256,
            num_beams=4,
        )

    return tokenizer.batch_decode(
        output,
        skip_special_tokens=True,
    )[0].strip()


translated_segments = []

for i, segment in enumerate(segments):
    translated_text = translate_text(segment["text"])

    translated_segments.append(
        {
            **segment,
            "translated_text": translated_text,
        }
    )

    print(
        f"Translated {i + 1}/{len(segments)}: "
        f"{translated_text}"
    )

with open(TRANSLATED_JSON, "w", encoding="utf-8") as f:
    json.dump(
        translated_segments,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Translated transcript saved:", TRANSLATED_JSON)

del translator
del tokenizer

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ==================================================
# 6. GENERATE SPEECH USING EDGE TTS
# ==================================================

async def generate_speech():
    generated = []

    for i, segment in enumerate(translated_segments):
        text = segment["translated_text"].strip()

        if not text:
            continue

        mp3_path = OUTPUT_DIR / f"segment_{i:04d}.mp3"
        wav_path = OUTPUT_DIR / f"segment_{i:04d}.wav"

        await edge_tts.Communicate(
            text=text,
            voice=VOICE,
        ).save(str(mp3_path))

        subprocess.run(
            [
                FFMPEG, "-y",
                "-i", str(mp3_path),
                "-ac", "1",
                "-ar", str(SAMPLE_RATE),
                "-c:a", "pcm_s16le",
                str(wav_path),
            ],
            check=True,
            capture_output=True,
        )

        speech, sr = sf.read(
            wav_path,
            dtype="float32",
        )

        if speech.ndim > 1:
            speech = speech.mean(axis=1)

        generated.append(
            {
                "start": segment["start"],
                "end": segment["end"],
                "audio": speech,
            }
        )

        print(
            f"Generated speech {i + 1}/"
            f"{len(translated_segments)}"
        )

    return generated


generated_segments = await generate_speech()

if not generated_segments:
    raise RuntimeError("No translated speech was generated.")

# ==================================================
# 7. ALIGN GENERATED SPEECH TO THE VIDEO TIMELINE
# ==================================================

probe = subprocess.run(
    [FFMPEG, "-hide_banner", "-i", str(VIDEO_PATH)],
    capture_output=True,
    text=True,
)

match = re.search(
    r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)",
    probe.stderr,
)

if not match:
    raise RuntimeError("Could not determine video duration.")

duration = (
    int(match.group(1)) * 3600
    + int(match.group(2)) * 60
    + float(match.group(3))
)

total_samples = int(duration * SAMPLE_RATE)
timeline = np.zeros(total_samples, dtype=np.float32)

for item in generated_segments:
    start = max(0, int(item["start"] * SAMPLE_RATE))
    speech = item["audio"]

    if start >= total_samples:
        continue

    end = min(start + len(speech), total_samples)
    speech = speech[: end - start]

    # Mix overlapping speech rather than allowing clipping.
    timeline[start:end] += speech

peak = np.max(np.abs(timeline)) if len(timeline) else 0

if peak > 0.99:
    timeline *= 0.99 / peak

sf.write(DUBBED_AUDIO, timeline, SAMPLE_RATE)

print("Translated audio saved:", DUBBED_AUDIO)

# ==================================================
# 8. MERGE TRANSLATED AUDIO WITH VIDEO
# ==================================================

subprocess.run(
    [
        FFMPEG, "-y",
        "-i", str(VIDEO_PATH),
        "-i", str(DUBBED_AUDIO),
        "-map", "0:v:0",
        "-map", "1:a:0",
        "-c:v", "copy",
        "-c:a", "aac",
        "-b:a", "192k",
        "-t", str(duration),
        "-movflags", "+faststart",
        str(FINAL_VIDEO),
    ],
    check=True,
)

if not FINAL_VIDEO.is_file() or FINAL_VIDEO.stat().st_size == 0:
    raise RuntimeError("The final translated video was not created.")

print("\n" + "=" * 55)
print("VOCULT TRANSLATION COMPLETED")
print("=" * 55)
print("Language:", LANGUAGE_NAME)
print("Final video:", FINAL_VIDEO)
print("Transcript:", TRANSLATED_JSON)
print("Audio:", DUBBED_AUDIO)
print("Output folder:", OUTPUT_DIR)

from IPython.display import display, Video, FileLink

display(Video(filename=str(FINAL_VIDEO), embed=False, width=720))
display(FileLink(str(FINAL_VIDEO)))


VOCULT — AI VIDEO TRANSLATOR
Video directory: /Users/chaitanyabhardwaj/vocult/files
Device: cpu



Enter video filename (e.g., test_video.mp4):  Chaitanya Video Assignment.mp4



Select the target language:

1. Hindi
2. English
3. Spanish
4. French
5. German
6. Japanese
7. Chinese (Simplified)
8. Arabic
9. Bengali
10. Tamil
11. Telugu
12. Punjabi
13. Urdu



Enter language number:  12



TRANSLATION SETTINGS
Input video : /Users/chaitanyabhardwaj/vocult/files/Chaitanya Video Assignment.mp4
Target      : Punjabi
NLLB code   : pan_Guru
TTS voice   : pa-IN-VaaniNeural
Output      : /Users/chaitanyabhardwaj/vocult/files/translated_videos/Chaitanya Video Assignment_pan_Guru/Chaitanya Video Assignment_pan_Guru.mp4


ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 13.1.6 (clang-1316.0.21.2.5)
  configuration: --prefix=/Volumes/tempdisk/sw --extra-cflags=-fno-stack-check --arch=arm64 --cc=/usr/bin/clang --enable-gpl --enable-libvmaf --enable-libopenjpeg --enable-libopus --enable-libmp3lame --enable-libx264 --enable-libx265 --enable-libvpx --enable-libwebp --enable-libass --enable-libfreetype --enable-fontconfig --enable-libtheora --enable-libvorbis --enable-libsnappy --enable-libaom --enable-libvidstab --enable-libzimg --enable-libsvtav1 --enable-libharfbuzz --enable-libkvazaar --pkg-config-flags=--static --enable-ffplay --enable-postproc --enable-neon --enable-runtime-cpudetect --disable-indev=qtkit --disable-indev=x11grab_xcb
  libavutil      59. 39.100 / 59. 39.100
  libavcodec     61. 19.100 / 61. 19.100
  libavformat    61.  7.100 / 61.  7.100
  libavdevice    61.  3.100 / 61.  3.100
  libavfilter    10.  4.100 / 10.  4.100
  libswscale      8. 


Original audio extracted.
2026-10-22 19:58:00 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-10-22 19:58:00 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/whisperx/assets/pytorch_model.bin`


2026-10-22 19:58:01 - whisperx.asr - WARNING - Audio is shorter than 30s, language detection may be inaccurate
2026-10-22 19:58:02 - whisperx.asr - INFO - Detected language: en (0.75) in first 30s of audio
Detected source language: en
Transcribed 1 speech segments.

Loading translation model...
